# 9. 딥러닝 임베딩 모델 입문 실습

> 이 노트북은 강의 자료에서 자동 변환되었습니다.  
> Google Colab에서 `런타임 > 런타임 유형 변경 > GPU`로 설정 후 실행하세요.


# 9. 딥러닝 임베딩 모델 입문 실습

## 학습 목표

1. 원-핫 인코딩과 임베딩의 차이를 설명할 수 있다.
2. `nn.Embedding`의 입력/출력 shape를 읽을 수 있다.
3. `padding_idx`가 왜 필요한지 이해할 수 있다.
4. `EmbeddingBag`이 어떤 상황에서 유용한지 감을 잡는다.
5. 간단한 텍스트 분류기를 학습시키며 임베딩이 어떻게 함께 학습되는지 본다.
6. 코사인 유사도로 단어/문장 벡터를 비교할 수 있다.


```text
단어 -> 정수 id -> 임베딩 테이블 조회 -> 밀집 벡터
밀집 벡터 -> 평균/합/모델 통과 -> 문장 표현 -> 분류/검색/추천
```


---

## 진행 순서

1. [왜 임베딩이 필요할까?](#part1) - 원-핫 인코딩의 한계
2. [임베딩은 무엇일까?](#part2) - nn.Embedding과 lookup table
3. [Shape 읽기와 padding_idx](#part3) - 입출력 shape와 패딩 처리
4. [EmbeddingBag](#part4) - 토큰 묶음을 한 벡터로 요약
5. [임베딩은 어떻게 의미를 배우는가?](#part5) - 감성 분류와 함께 학습
6. [코사인 유사도로 비교하기](#part6) - 학습된 단어 벡터 비교
7. [2차원으로 도식화해 보기](#part7) - PCA 투영 시각화
8. [문장 임베딩 맛보기](#part8) - 단어 임베딩 평균으로 문장 벡터
9. [통합 정리](#part9) - 마무리 체크리스트와 확장 과제

---
### 환경 설정


In [ ]:
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import math
import random
import subprocess
from collections import Counter

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colab 한글 폰트 설정
if 'COLAB_RELEASE_TAG' in os.environ:
    subprocess.run(['apt-get', '-qq', '-y', 'install', 'fonts-nanum'], capture_output=True)
    import matplotlib.font_manager as fm
    for f in fm.findSystemFonts(['/usr/share/fonts/truetype/nanum']):
        fm.fontManager.addfont(f)
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

torch.manual_seed(7)
random.seed(7)

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 1. 왜 임베딩이 필요할까?

**학습목표**: 원-핫 인코딩의 원리와 한계를 설명할 수 있다

먼저 텍스트를 숫자로 바꾸는 가장 단순한 방법은 **원-핫 인코딩**입니다.

예를 들어 단어 사전이 5개라면:

- `사과` -> `[1, 0, 0, 0, 0]`
- `바나나` -> `[0, 1, 0, 0, 0]`
- `자동차` -> `[0, 0, 1, 0, 0]`

원-핫은 단순하지만 한계가 큽니다.

- 단어 수가 많아지면 벡터가 너무 커집니다.
- 서로 다른 단어는 거의 모두 `직교`처럼 보입니다.
- `사과`와 `바나나`가 둘 다 과일이라는 정보가 벡터 안에 들어있지 않습니다.

비유하면, 원-핫은 학생 이름표만 주고 서로의 관계는 전혀 적어주지 않은 상태와 비슷합니다.


In [ ]:
vocab = ['사과', '바나나', '자동차', '버스', '포도']
word_to_id = {word: idx for idx, word in enumerate(vocab)}
ids = torch.tensor([word_to_id[word] for word in vocab])
one_hot = F.one_hot(ids, num_classes=len(vocab)).float()

print('단어 사전:', vocab)
print('원-핫 shape:', one_hot.shape)
print()
print('사과 one-hot:', one_hot[word_to_id['사과']])
print('바나나 one-hot:', one_hot[word_to_id['바나나']])
print()
print('사과 vs 바나나 cosine:', F.cosine_similarity(one_hot[0].unsqueeze(0), one_hot[1].unsqueeze(0)).item())
print('사과 vs 사과 cosine  :', F.cosine_similarity(one_hot[0].unsqueeze(0), one_hot[0].unsqueeze(0)).item())

**핵심**: 원-핫은 단순하지만 차원 폭발, 의미 부재, 순서 무시라는 한계가 있다. 비슷한 단어를 비슷한 숫자로 표현할 수 없다.

---

## 2. 임베딩은 무엇일까?

**학습목표**: nn.Embedding의 역할과 lookup table 개념을 이해할 수 있다

임베딩은 **큰 희소 벡터 대신 작은 밀집 벡터로 대상을 표현하는 방법**입니다.

쉽게 말하면:

- 원-핫은 `번호표`
- 임베딩은 `의미가 반영된 좌표`

입니다.

예를 들어 단어를 2차원 지도 위에 찍는다고 생각해보면:

```text
의미가 비슷한 단어는 가까이
의미가 다른 단어는 멀리
```

가 되도록 좌표를 배우는 것이 임베딩의 핵심입니다.

PyTorch의 `nn.Embedding`은 이 좌표표, 즉 **학습 가능한 lookup table**를 만듭니다.


In [ ]:
embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=3)
lookup_ids = torch.tensor([0, 1, 4])
lookup_vectors = embedding(lookup_ids)

print('임베딩 테이블 weight shape:', embedding.weight.shape)
print('조회할 단어 id:', lookup_ids)
print('조회 결과 shape:', lookup_vectors.shape)
print('조회 결과:')
print(lookup_vectors)

**핵심**: 임베딩은 큰 희소 벡터 대신 작은 밀집 벡터로 대상을 표현한다. nn.Embedding은 학습 가능한 lookup table이다.

---

## 3. Shape 읽기와 `padding_idx`

**학습목표**: nn.Embedding의 입출력 shape와 padding_idx의 용도를 읽을 수 있다

공식 문서 기준으로 `nn.Embedding`은 다음처럼 생각하면 쉽습니다.

- 입력: 정수 인덱스 텐서 `(*)`
- 출력: 입력 shape 뒤에 `embedding_dim`이 하나 더 붙은 텐서 `(*, embedding_dim)`

즉 입력이 `(batch, length)`면 출력은 `(batch, length, embedding_dim)`이 됩니다.

또한 텍스트 배치에서는 길이를 맞추기 위해 `pad` 토큰을 자주 씁니다.
이때 `padding_idx`를 지정하면 해당 행은 학습 중 업데이트되지 않는 고정 패딩으로 다룰 수 있습니다.


In [ ]:
pad_embedding = nn.Embedding(num_embeddings=8, embedding_dim=4, padding_idx=0)
input_ids = torch.tensor([
    [0, 2, 3, 0],
    [1, 4, 5, 6],
], dtype=torch.long)
output_vectors = pad_embedding(input_ids)

print('입력 shape:', input_ids.shape)
print('출력 shape:', output_vectors.shape)
print()
print('padding row (index 0):')
print(pad_embedding.weight[0])
print()
print('첫 번째 샘플 임베딩:')
print(output_vectors[0])

**핵심**: 입력이 (batch, length)면 출력은 (batch, length, embedding_dim)이다. padding_idx를 지정하면 해당 행은 학습 중 업데이트되지 않는다.

---

## 4. `EmbeddingBag`은 언제 쓰면 좋을까?

**학습목표**: EmbeddingBag이 유용한 상황을 설명할 수 있다

`EmbeddingBag`은 여러 토큰의 임베딩을 **합(sum)**, **평균(mean)**, **최대(max)** 로 바로 모아주는 층입니다.

비유하면,

- `Embedding`: 단어별 좌표를 하나씩 꺼내기
- `EmbeddingBag`: 꺼낸 좌표들을 바로 한 문장 요약 벡터로 합치기

문서 분류, 추천, 광고처럼 `토큰 묶음 -> 하나의 벡터`가 자주 필요한 문제에서 유용합니다.


In [ ]:
bag = nn.EmbeddingBag(num_embeddings=10, embedding_dim=4, mode='mean', padding_idx=0)
bag_input = torch.tensor([
    [1, 2, 3, 0],
    [4, 5, 0, 0],
], dtype=torch.long)
bag_output = bag(bag_input)

print('EmbeddingBag 입력 shape:', bag_input.shape)
print('EmbeddingBag 출력 shape:', bag_output.shape)
print('각 문장 요약 벡터:')
print(bag_output)

**핵심**: EmbeddingBag은 여러 토큰의 임베딩을 바로 합/평균/최대로 모아준다. 문서 분류, 추천 등에서 유용하다.

---

## 5. 임베딩은 어떻게 "의미"를 배우는가?

**학습목표**: 임베딩이 과제를 풀면서 함께 학습되는 과정을 관찰할 수 있다

임베딩은 보통 **혼자서 의미를 배우는 것이 아니라, 어떤 과제를 풀면서 함께 학습**됩니다.

예를 들어 감성 분류를 시키면,

- 긍정 문장에서 자주 함께 쓰이는 단어들은 비슷한 방향으로 조정되고
- 부정 문장에서 자주 함께 쓰이는 단어들도 또 다른 방향으로 조정될 수 있습니다.

즉 임베딩은 `문제를 잘 풀기 위해` 업데이트되면서 점점 유용한 좌표가 됩니다.

이번에는 아주 작은 장난감 데이터셋으로 이 과정을 직접 봅니다.


In [ ]:
train_samples = [
    ('정말 좋다', 1),
    ('아주 좋다', 1),
    ('재미 있다', 1),
    ('추천 한다', 1),
    ('마음 에 든다', 1),
    ('최고 다', 1),
    ('너무 좋다', 1),
    ('재미 최고', 1),
    ('정말 싫다', 0),
    ('아주 별로', 0),
    ('지루 하다', 0),
    ('추천 안 한다', 0),
    ('마음 에 안 든다', 0),
    ('최악 이다', 0),
    ('너무 싫다', 0),
    ('지루 별로', 0),
]

PAD = '<pad>'
UNK = '<unk>'

def tokenize(text):
    return text.split()

counter = Counter()
for text, _ in train_samples:
    counter.update(tokenize(text))

vocab = [PAD, UNK] + sorted(counter.keys())
word_to_id = {word: idx for idx, word in enumerate(vocab)}
id_to_word = {idx: word for word, idx in word_to_id.items()}

def encode(text):
    return [word_to_id.get(token, word_to_id[UNK]) for token in tokenize(text)]

encoded = [encode(text) for text, _ in train_samples]
labels = torch.tensor([label for _, label in train_samples], dtype=torch.long)
max_len = max(len(seq) for seq in encoded)

padded = []
for seq in encoded:
    padded.append(seq + [word_to_id[PAD]] * (max_len - len(seq)))

inputs = torch.tensor(padded, dtype=torch.long)

class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_classes, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)
        self.classifier = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        mask = (x != word_to_id[PAD]).unsqueeze(-1)
        masked_emb = emb * mask
        lengths = mask.sum(dim=1).clamp(min=1)
        pooled = masked_emb.sum(dim=1) / lengths
        logits = self.classifier(pooled)
        return logits, pooled

model = MeanEmbeddingClassifier(vocab_size=len(vocab), emb_dim=8, num_classes=2, padding_idx=word_to_id[PAD])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

loss_history = []
for epoch in range(200):
    optimizer.zero_grad()
    logits, pooled = model(inputs)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

with torch.no_grad():
    logits, pooled = model(inputs)
    preds = logits.argmax(dim=1)
    accuracy = (preds == labels).float().mean().item()

print('학습 데이터 shape:', inputs.shape)
print('어휘 수:', len(vocab))
print('최대 문장 길이:', max_len)
print('최종 loss:', round(loss_history[-1], 4))
print('학습 정확도:', round(accuracy, 4))
print()
for idx in range(4):
    print(train_samples[idx][0], '-> 예측', preds[idx].item(), '| 정답', labels[idx].item())

**핵심**: 임베딩은 혼자 의미를 배우는 것이 아니라, 분류 등의 과제를 풀면서 함께 업데이트된다.

---

## 6. 학습된 단어 임베딩을 코사인 유사도로 비교하기

**학습목표**: 코사인 유사도로 학습된 단어 벡터 간 관계를 비교할 수 있다

이제 같은 임베딩 공간 안에서 단어 간 거리를 비교해볼 수 있습니다.

코사인 유사도는 방향이 비슷한지를 보는 값입니다.

- `1`에 가까우면 매우 비슷한 방향
- `0`에 가까우면 관련성이 약함
- `-1`에 가까우면 반대 방향

작은 장난감 데이터셋이라 완벽한 의미 공간은 아니지만, **학습 목표에 따라 단어 좌표가 달라진다**는 점을 보여주기에는 충분합니다.


In [ ]:
def cosine_between(word_a, word_b):
    idx_a = word_to_id[word_a]
    idx_b = word_to_id[word_b]
    vec_a = model.embedding.weight[idx_a].unsqueeze(0)
    vec_b = model.embedding.weight[idx_b].unsqueeze(0)
    return F.cosine_similarity(vec_a, vec_b).item()

pairs = [
    ('좋다', '최고'),
    ('좋다', '추천'),
    ('싫다', '별로'),
    ('좋다', '싫다'),
    ('재미', '지루'),
]

for a, b in pairs:
    print(f'{a:>4} vs {b:<4} -> cosine = {cosine_between(a, b):.4f}')

print()

interesting_words = ['좋다', '최고', '추천', '싫다', '별로', '지루']
weights = model.embedding.weight.detach()
for word in interesting_words:
    idx = word_to_id[word]
    sims = F.cosine_similarity(weights[idx].unsqueeze(0), weights, dim=1)
    topk = torch.topk(sims, k=4).indices.tolist()
    neighbors = [id_to_word[i] for i in topk if id_to_word[i] not in {PAD, word}][:3]
    print(f'{word:>4}의 가까운 단어:', neighbors)

**핵심**: 코사인 유사도가 1에 가까우면 비슷한 방향, -1에 가까우면 반대 방향이다. 학습 목표에 따라 단어 좌표가 달라진다.

---

## 7. 2차원으로 도식화해 보기

**학습목표**: 고차원 임베딩을 2차원으로 투영하여 시각적으로 확인할 수 있다

임베딩은 원래 8차원 공간에 있지만, 사람 눈으로 보려면 2차원으로 눌러서 봐야 합니다.

여기서는 `torch.pca_lowrank`를 사용해 대략적인 2차원 배치를 그립니다.

주의할 점:

- 2차원 투영은 정보 손실이 있습니다.
- 그림은 감을 잡는 용도이지, 절대적인 진실은 아닙니다.


In [ ]:
plot_words = ['좋다', '최고', '추천', '재미', '싫다', '별로', '지루', '최악']
indices = torch.tensor([word_to_id[word] for word in plot_words])
points = model.embedding.weight.detach()[indices]
centered = points - points.mean(dim=0, keepdim=True)
U, S, V = torch.pca_lowrank(centered, q=2)
coords = centered @ V[:, :2]

plt.figure(figsize=(7, 5))
plt.axhline(0, color='lightgray', linewidth=1)
plt.axvline(0, color='lightgray', linewidth=1)
for i, word in enumerate(plot_words):
    x = coords[i, 0].item()
    y = coords[i, 1].item()
    plt.scatter(x, y, s=80)
    plt.text(x + 0.02, y + 0.02, word, fontsize=10)
plt.title('학습된 단어 임베딩의 2D 투영')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

**핵심**: 2차원 투영은 감을 잡는 용도이지 절대적인 진실은 아니다. 정보 손실이 있음을 기억해야 한다.

---

## 8. 문장 임베딩의 아주 작은 맛보기

**학습목표**: 단어 임베딩 평균으로 간단한 문장 벡터를 만들 수 있다

문장 임베딩은 단어 임베딩을 바탕으로 문장 전체를 하나의 벡터로 표현하는 것입니다.

이번 실습에서는 복잡한 트랜스포머 대신, **단어 임베딩 평균**으로 매우 단순한 문장 벡터를 만들어 봅니다.

현업에서는 보통 SBERT 같은 사전학습 문장 임베딩 모델을 많이 사용하지만, 지금 단계에서는 `문장도 결국 벡터가 될 수 있다`는 감을 잡는 것이 중요합니다.


In [ ]:
test_sentences = [
    '정말 좋다',
    '아주 좋다',
    '정말 싫다',
    '지루 하다',
    '추천 한다',
]

def encode_batch(texts):
    encoded = [encode(text) for text in texts]
    max_len = max(len(seq) for seq in encoded)
    padded = [seq + [word_to_id[PAD]] * (max_len - len(seq)) for seq in encoded]
    return torch.tensor(padded, dtype=torch.long)

with torch.no_grad():
    batch = encode_batch(test_sentences)
    logits, sentence_vectors = model(batch)
    sim_matrix = F.cosine_similarity(
        sentence_vectors.unsqueeze(1),
        sentence_vectors.unsqueeze(0),
        dim=2,
    )

print('문장 벡터 shape:', sentence_vectors.shape)
print()
print('문장 유사도 행렬')
print(torch.round(sim_matrix * 100) / 100)
print()
for i, text in enumerate(test_sentences):
    print(i, text)

**핵심**: 현업에서는 SBERT 같은 사전학습 문장 임베딩 모델을 사용하지만, 핵심 원리는 단어 벡터의 조합이다.

---

## 9. 통합 정리

아래 질문에 답할 수 있으면 오늘 핵심은 잡은 것입니다.

1. 원-핫 인코딩의 가장 큰 한계는 무엇인가?
2. `nn.Embedding`은 왜 lookup table이라고 불리는가?
3. 입력이 `(batch, length)`일 때 출력 shape는 왜 `(batch, length, embedding_dim)`인가?
4. `padding_idx`는 무엇을 위해 쓰는가?
5. 임베딩은 왜 분류/검색/추천 문제에서 유용한가?

#### 선택 확장 과제

여기까지가 `입문 Colab`의 핵심 범위입니다. 이후에는 다음 방향으로 확장할 수 있습니다.

**확장 1. Word2Vec, GloVe, FastText**

- `왜 임베딩이 필요한가`를 넘어서
- `임베딩을 어떻게 더 잘 학습할까`로 넘어가는 단계입니다.

**확장 2. 사전학습 임베딩과 전이학습**

- 이미 학습된 단어 벡터를 불러와 재사용
- 작은 데이터셋에서도 더 좋은 출발점 확보

**확장 3. 문장 임베딩과 SBERT**

- 문장 전체를 한 번에 임베딩
- 유사도 계산, semantic search, 추천, RAG 검색기에 연결 가능

**확장 4. 추천 시스템과 범주형 피처 임베딩**

- 임베딩은 NLP뿐 아니라 추천/광고/탭 데이터에서도 핵심 도구입니다.
- 사용자 id, 상품 id, 카테고리 id도 임베딩으로 표현할 수 있습니다.

#### 참고

- 위키독스 책: [딥 러닝 파이토치 교과서 - 입문부터 LLM 파인튜닝까지](https://wikidocs.net/book/2788)
- PyTorch 공식 문서: [`torch.nn.Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
- PyTorch 공식 문서: [`torch.nn.EmbeddingBag`](https://docs.pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html)
- PyTorch 공식 문서: [`torch.nn.functional.one_hot`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)
- PyTorch 공식 문서: [`torch.nn.functional.cosine_similarity`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cosine_similarity.html)
- Sentence Transformers 공식 문서: [README / Quickstart](https://www.sbert.net/README.html)
